# 03a - Fine-Tune Edilmemiş Modeller Diagnostic Deneyi

Bu notebook, S&P 500 annotation Excel dosyalarını okuyup fine-tune edilmemiş taban modelleri aynı dış test setinde değerlendirir.

Test edilen modeller:

- `bert-base-uncased`
- `distilbert-base-uncased`
- `roberta-base`

Not: Bu modeller duygu analizi için fine-tune edilmiş modeller değildir. Bu yüzden 3 sınıflı classification head rastgele başlatılır. Notebook her modeli yalnızca 1 kez çalıştırır ve sonuçları sadece ekranda gösterir. Herhangi bir sonuç dosyası kaydetmez.


In [1]:
from thesis_utils import APP_ROOT, DATA_ROOT, PREVIEW_ROWS, PROJECT_ROOT, paths
# ============================================================
# 1) IMPORTLAR VE AYARLAR
# ============================================================

from pathlib import Path
import re
import gc
import warnings

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    set_seed,
    logging as hf_logging
)

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()

# Excel dosyalarının bulunduğu klasör
ANNOTATION_DIR = paths.SP500_HUMAN_REVIEW_BATCHES_DIR

# Okunacak dosya örüntüsü
BATCH_PATTERN = "SP500_annotation_batch_*.xlsx"

# Fine-tune edilmemiş taban modeller
BASE_MODELS = {
    "bert_base_uncased_unfinetuned": "bert-base-uncased",
    "distilbert_base_uncased_unfinetuned": "distilbert-base-uncased",
    "roberta_base_unfinetuned": "roberta-base",
}

# Etiket sırası
VALID_LABELS = ["negative", "neutral", "positive"]
LABEL2ID = {label: i for i, label in enumerate(VALID_LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

MAX_LENGTH = 128
BATCH_SIZE = 64

# Her model 1 kere çalışır. Classification head'in rastgele başlangıcı sabit olsun diye tek sabit başlangıç kullanılır.
RANDOM_STATE = 42

# Cihaz
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Annotation dir:", ANNOTATION_DIR)
print("Annotation dir exists:", ANNOTATION_DIR.exists())
print("Device:", device)
print("Batch size:", BATCH_SIZE)

if not ANNOTATION_DIR.exists():
    raise FileNotFoundError(f"Klasör bulunamadı: {ANNOTATION_DIR}")

Annotation dir: D:\serkan.kaymak\financial_sentiment_thesis\db\annotations\sp500_human_review_batches
Annotation dir exists: True
Device: cpu
Batch size: 64


In [2]:
# ============================================================
# 2) EXCEL DOSYALARINI OKU
# ============================================================

def normalize_colname(c):
    c = str(c).strip()
    c = re.sub(r"\s+", "_", c)
    return c


def find_header_row(raw_df, required_cols=("annotation_id", "sample_id", "text_en")):
    """Başlık satırı ilk birkaç satırdan hangisiyse otomatik bulur."""
    for idx in range(min(10, len(raw_df))):
        row_values = raw_df.iloc[idx].astype(str).str.strip().str.lower().tolist()
        hits = sum(col in row_values for col in required_cols)
        if hits >= 2:
            return idx
    return 0


def read_annotation_excel(path):
    path = Path(path)

    raw = pd.read_excel(path, header=None)
    header_row = find_header_row(raw)

    df = pd.read_excel(path, header=header_row)
    df.columns = [normalize_colname(c) for c in df.columns]

    # Tamamen boş satırları at
    df = df.dropna(how="all").copy()

    # Dosya bilgisini ekle
    df["source_file"] = path.name

    # Gereksiz kolonları at
    drop_cols = []
    for c in df.columns:
        lc = str(c).lower()
        if lc.startswith("unnamed"):
            drop_cols.append(c)
        if c in ["Özet", "Değer", "Ozet", "Deger"]:
            drop_cols.append(c)

    df = df.drop(columns=list(set(drop_cols)), errors="ignore")
    return df


# Klasördeki Excel dosyalarını kontrol et
all_xlsx = sorted([
    f for f in ANNOTATION_DIR.glob("*.xlsx")
    if not f.name.startswith("~$")
])

print("Klasördeki toplam .xlsx dosya sayısı:", len(all_xlsx))
print("İlk 3 .xlsx dosyası:")
for f in all_xlsx[:PREVIEW_ROWS]:
    print(" -", f.name)

# Pattern'e uyan batch dosyalarını bul
excel_files = sorted([
    f for f in ANNOTATION_DIR.glob(BATCH_PATTERN)
    if not f.name.startswith("~$")
])

print("\nPattern'e uyan Excel dosyası sayısı:", len(excel_files))
print("Pattern:", BATCH_PATTERN)

for f in excel_files[:PREVIEW_ROWS]:
    print(" -", f.name)

if len(excel_files) == 0:
    raise FileNotFoundError(
        f"Hiç Excel dosyası bulunamadı. Klasör: {ANNOTATION_DIR} | Pattern: {BATCH_PATTERN}"
    )

all_parts = []
failed_files = []

for f in tqdm(excel_files, desc="Excel dosyaları okunuyor"):
    try:
        all_parts.append(read_annotation_excel(f))
    except Exception as e:
        failed_files.append((f.name, str(e)))

if len(all_parts) == 0:
    raise RuntimeError("Hiçbir Excel dosyası okunamadı.")

annotation_all_df = pd.concat(all_parts, ignore_index=True)

print("\nannotation_all_df shape:", annotation_all_df.shape)
print("Başarısız dosya sayısı:", len(failed_files))

if failed_files:
    display(pd.DataFrame(failed_files, columns=["file", "error"]))

print("\nKolonlar:")
print(annotation_all_df.columns.tolist())

display(annotation_all_df.head(PREVIEW_ROWS))

Klasördeki toplam .xlsx dosya sayısı: 106
İlk 3 .xlsx dosyası:
 - SP500_annotation_batch_001.xlsx
 - SP500_annotation_batch_002.xlsx
 - SP500_annotation_batch_004.xlsx

Pattern'e uyan Excel dosyası sayısı: 106
Pattern: SP500_annotation_batch_*.xlsx
 - SP500_annotation_batch_001.xlsx
 - SP500_annotation_batch_002.xlsx
 - SP500_annotation_batch_004.xlsx


Excel dosyaları okunuyor:   0%|          | 0/106 [00:00<?, ?it/s]


annotation_all_df shape: (1060, 14)
Başarısız dosya sayısı: 0

Kolonlar:
['annotation_id', 'sample_id', 'date', 'text_en', 'text_tr', 'finbert_label', 'finbert_confidence', 'chatgpt_label', 'chatgpt_confidence', 'chatgpt_reason_tr', 'finbert_correctness', 'finbert_correctness_note', 'source_file', 'final_label']


,annotation_id,sample_id,date,text_en,text_tr,finbert_label,finbert_confidence,chatgpt_label,chatgpt_confidence,chatgpt_reason_tr,finbert_correctness,finbert_correctness_note,source_file,final_label
0,SP500_ANN_0001,SP500_HEAD_000004,2008-01-03,"U.S. Stocks Higher After Economic Data, Monsan...",ABD hisseleri ekonomik veriler ve Monsanto gör...,positive,0.861627,positive,high,ABD hisseleri yükseliyor; piyasa açısından olu...,correct,FinBERT etiketi final etiket ile aynı.,SP500_annotation_batch_001.xlsx,NaN
1,SP500_ANN_0002,SP500_HEAD_016918,2023-12-15,Stock Market Outlook 2024: Rare Bullish Signal...,2024 borsa görünümü: Nadir bir boğa sinyali S&...,positive,0.881073,positive,high,Bullish sinyal ve S&P 500'de güçlü yükseliş be...,correct,FinBERT etiketi final etiket ile aynı.,SP500_annotation_batch_001.xlsx,NaN
2,SP500_ANN_0003,SP500_HEAD_013289,2023-03-20,Federal Reserve Rate Hike Odds Grow As Bank-Cr...,Banka krizi korkuları azalırken Fed faiz artır...,negative,0.625434,positive,medium,Başlık karışık olsa da banka krizi korkularını...,wrong,FinBERT negative demiş; final etiket positive....,SP500_annotation_batch_001.xlsx,NaN


In [3]:
# ============================================================
# 3) TEST SETİNİ HAZIRLA
# ============================================================

required_cols = ["text_en", "chatgpt_label"]
missing_cols = [c for c in required_cols if c not in annotation_all_df.columns]
if missing_cols:
    raise ValueError(f"Eksik zorunlu kolonlar: {missing_cols}")

# Gold label:
# 1) final_label doluysa onu kullan
# 2) final_label boşsa chatgpt_label kullan
if "final_label" in annotation_all_df.columns:
    final_label_clean = annotation_all_df["final_label"].astype("string").str.lower().str.strip()
    chatgpt_label_clean = annotation_all_df["chatgpt_label"].astype("string").str.lower().str.strip()
    gold_label = final_label_clean.where(final_label_clean.notna() & (final_label_clean != ""), chatgpt_label_clean)
else:
    gold_label = annotation_all_df["chatgpt_label"].astype("string").str.lower().str.strip()

# Metin ve etiketleri temizle
test_df = annotation_all_df.copy()
test_df["gold_label"] = gold_label
test_df["text_en"] = test_df["text_en"].astype("string").str.strip()

test_df = test_df.dropna(subset=["text_en", "gold_label"]).copy()
test_df = test_df[test_df["text_en"] != ""].copy()
test_df = test_df[test_df["gold_label"].isin(VALID_LABELS)].copy()

test_df = test_df.reset_index(drop=True)

print("test_df shape:", test_df.shape)
print("Kullanılan metin kolonu: text_en")
print("Kullanılan gold label kolonu: final_label doluysa final_label, boşsa chatgpt_label")

print("\nGold label dağılımı - adet:")
display(test_df["gold_label"].value_counts().reindex(VALID_LABELS).to_frame("count"))

print("Gold label dağılımı - yüzde:")
display((test_df["gold_label"].value_counts(normalize=True).reindex(VALID_LABELS) * 100).round(2).to_frame("percent"))

display(test_df[["annotation_id", "sample_id", "text_en", "finbert_label", "chatgpt_label", "final_label", "gold_label", "source_file"]].head(PREVIEW_ROWS))

test_df shape: (1060, 15)
Kullanılan metin kolonu: text_en
Kullanılan gold label kolonu: final_label doluysa final_label, boşsa chatgpt_label

Gold label dağılımı - adet:


,count
gold_label,
negative,323
neutral,339
positive,398


Gold label dağılımı - yüzde:


,percent
gold_label,
negative,30.47
neutral,31.98
positive,37.55


,annotation_id,sample_id,text_en,finbert_label,chatgpt_label,final_label,gold_label,source_file
0,SP500_ANN_0001,SP500_HEAD_000004,"U.S. Stocks Higher After Economic Data, Monsan...",positive,positive,NaN,positive,SP500_annotation_batch_001.xlsx
1,SP500_ANN_0002,SP500_HEAD_016918,Stock Market Outlook 2024: Rare Bullish Signal...,positive,positive,NaN,positive,SP500_annotation_batch_001.xlsx
2,SP500_ANN_0003,SP500_HEAD_013289,Federal Reserve Rate Hike Odds Grow As Bank-Cr...,negative,positive,NaN,positive,SP500_annotation_batch_001.xlsx


In [4]:
# ============================================================
# 4) YARDIMCI FONKSİYONLAR
# ============================================================

def compute_metrics(model_name, y_true, y_pred):
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    return {
        "model": model_name,
        "n_eval": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
    }


def predict_unfinetuned_model(model_id, texts):
    """Fine-tune edilmemiş modeli 3 sınıflı classification head ile 1 kere çalıştırır."""

    # Her model için aynı sabit başlangıç kullanılır; seed döngüsü yoktur.
    set_seed(RANDOM_STATE)

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=3,
        id2label=ID2LABEL,
        label2id=LABEL2ID
    )

    model.to(device)
    model.eval()

    preds = []

    for start in tqdm(range(0, len(texts), BATCH_SIZE), desc=f"Predicting {model_id}"):
        batch_texts = texts[start:start + BATCH_SIZE]

        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            outputs = model(**enc)
            batch_preds = outputs.logits.argmax(dim=-1).detach().cpu().numpy()

        preds.extend(batch_preds)

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return np.array(preds)


def show_single_model_result(model_name, y_true, y_pred):
    result = compute_metrics(model_name, y_true, y_pred)
    result_df = pd.DataFrame([result]).round(4)

    display(result_df)

    print("Classification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=VALID_LABELS,
            zero_division=0,
            digits=4
        )
    )

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{x}" for x in VALID_LABELS],
        columns=[f"pred_{x}" for x in VALID_LABELS]
    )

    print("Confusion Matrix:")
    display(cm_df)

    return result

In [5]:
# ============================================================
# 5) FINE-TUNE EDİLMEMİŞ MODELLERİ TEST ET
# ============================================================

texts = test_df["text_en"].astype(str).tolist()
y_true_labels = test_df["gold_label"].astype(str).str.lower().str.strip().tolist()
y_true = np.array([LABEL2ID[label] for label in y_true_labels])

all_results = []

for model_name, model_id in BASE_MODELS.items():
    print("\n" + "=" * 100)
    print("MODEL TEST EDİLİYOR:", model_name)
    print("HF model id:", model_id)
    print("=" * 100)

    y_pred = predict_unfinetuned_model(model_id, texts)
    result = show_single_model_result(model_name, y_true, y_pred)
    all_results.append(result)

results_df = pd.DataFrame(all_results).round(4)

print("\n" + "=" * 100)
print("FINE-TUNE EDİLMEMİŞ MODELLER - ÖZET SONUÇ TABLOSU")
print("=" * 100)

display(results_df.sort_values("f1_macro", ascending=False).reset_index(drop=True))


MODEL TEST EDİLİYOR: bert_base_uncased_unfinetuned
HF model id: bert-base-uncased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Predicting bert-base-uncased:   0%|          | 0/17 [00:00<?, ?it/s]

,model,n_eval,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,bert_base_uncased_unfinetuned,1060,0.3113,0.2152,0.326,0.2034,0.2014,0.3113,0.1928


Classification Report:
              precision    recall  f1-score   support

    negative     0.3368    0.0991    0.1531       323
     neutral     0.3088    0.8791    0.4571       339
    positive     0.0000    0.0000    0.0000       398

    accuracy                         0.3113      1060
   macro avg     0.2152    0.3260    0.2034      1060
weighted avg     0.2014    0.3113    0.1928      1060

Confusion Matrix:


...

,model,n_eval,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,roberta_base_unfinetuned,1060,0.3047,0.1016,0.3333,0.1557,0.0929,0.3047,0.1423


Classification Report:
              precision    recall  f1-score   support

    negative     0.3047    1.0000    0.4671       323
     neutral     0.0000    0.0000    0.0000       339
    positive     0.0000    0.0000    0.0000       398

    accuracy                         0.3047      1060
   macro avg     0.1016    0.3333    0.1557      1060
weighted avg     0.0929    0.3047    0.1423      1060

Confusion Matrix:


,pred_negative,pred_neutral,pred_positive
true_negative,323,0,0
true_neutral,339,0,0
true_positive,398,0,0



FINE-TUNE EDİLMEMİŞ MODELLER - ÖZET SONUÇ TABLOSU


,model,n_eval,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,distilbert_base_uncased_unfinetuned,1060,0.3160,0.2662,0.3036,0.2576,0.2719,0.3160,0.2670
1,bert_base_uncased_unfinetuned,1060,0.3113,0.2152,0.3260,0.2034,0.2014,0.3113,0.1928
2,roberta_base_unfinetuned,1060,0.3047,0.1016,0.3333,0.1557,0.0929,0.3047,0.1423


## Yorumlama Notu

Bu sonuçlar, modellerin fine-tune edilmeden doğrudan finansal duygu sınıflandırma görevinde kullanılınca nasıl davrandığını gösterir. Bu modellerin classification head kısmı görev için eğitilmediğinden başarılarının düşük ve dengesiz çıkması normaldir. Fine-tune edilmiş modellerin yüksek başarısı, modelin finansal duygu sınıflandırma görevini öğrenmiş olduğunu gösterir.